# Benchmark Training Runner

This notebook is the training-only benchmark runner.
It trains configured models, calibrates on validation outputs, and writes deterministic artifacts for artifact-only evaluation.

All outputs are written under `ml_model/results/benchmarks/<dataset_version>_<timestamp>/` and consumed by the evaluation notebook.

In [ ]:
import gc
import math
import os
import random
import sys
import time
from contextlib import nullcontext
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch import autocast
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding, get_cosine_schedule_with_warmup


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ml_model.preprocessing.dataset_io import (
    build_split_hygiene_evidence,
    build_split_summaries,
    checkpoint_dir,
    encode_labels,
    evaluation_dir,
    load_data_splits,
    loss_variant_dir,
    make_output_dir,
    model_run_dir,
    resolve_data_dir,
    save_csv,
    save_json,
    save_numpy_artifacts,
    seed_run_dir,
)
from ml_model.training.losses import LOSS_ABLATION_GRID, build_loss, compute_class_weights
from ml_model.evaluation.metrics import (
    collect_logits_labels_loss,
    confidence_band_summary_frame,
    compute_per_class_metrics,
    estimate_inference_latency_ms,
    evaluate_from_logits,
    fit_temperature_scaling,
    model_size_megabytes,
    per_class_recall_at_threshold_frame,
    save_confusion_matrix_artifacts,
    save_reliability_diagram_artifacts,
    threshold_security_summary,
    top_label_calibration_frame,
)
from ml_model.training.model_factory import build_model, infer_architecture_family, infer_head_type

print(f"PyTorch     : {torch.__version__}")
print(f"CUDA avail  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
    print(f"VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("GPU         : CPU fallback mode")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CUDA_BF16 = torch.cuda.is_available() and hasattr(torch.cuda, "is_bf16_supported") and torch.cuda.is_bf16_supported()
print(f"AMP enabled : {CUDA_BF16}")

In [ ]:
TEXT_COL = "combined_payload"
LABEL_COL = "final_label"
EXPECTED_CLASSES = [
    "Code Injection",
    "Normal",
    "Other Attacks",
    "SQL Injection",
]

DATASET_VERSION = "v3_907k_cleaned"
BENCHMARK_SEEDS = [42, 1337, 2026]
N_EPOCHS = 5
EARLY_STOP_PATIENCE = 3
LOG_EVERY_STEPS = 200
MAX_GRAD_NORM = 1.0
ECE_N_BINS = 15

CHECKPOINT_SELECTION_RULE = "validation_macro_f1 (tie-break: validation_loss)"
MODEL_SELECTION_RULE = "highest validation_macro_f1_mean only across seeds and loss variants"

SMOKE_MODE = os.getenv("BENCHMARK_SMOKE_MODE", "").strip().lower() in {"1", "true", "yes", "on"}
SMOKE_MAX_SAMPLES_PER_CLASS = int(os.getenv("BENCHMARK_SMOKE_MAX_SAMPLES_PER_CLASS", "128"))
SMOKE_MODEL_KEYS = ["distilbert"]
SMOKE_SEEDS = [42]
SMOKE_LOSS_KEYS = ["ce"]

MODEL_REGISTRY = {
    "distilbert": {
        "model_key": "distilbert",
        "model_id": "distilbert-base-uncased",
        "architecture": "transformer",
        "experiment_phase": "controlled_backbone_benchmark",
        "learning_rate": 3e-5,
        "per_device_train_batch_size": 64,
        "gradient_accumulation_steps": 2,
        "effective_batch_size": 128,
        "weight_decay": 0.01,
        "dropout_prob": 0.25,
        "num_train_epochs": N_EPOCHS,
        "warmup_ratio": 0.04,
        "max_seq_len": 128,
        "head_hidden_dim": 256,
        "activation": "gelu",
        "focal_gamma": 2.0,
        "eval_batch_multiplier": 2,
    },
    "minilm_l6": {
        "model_key": "minilm_l6",
        "model_id": "nreimers/MiniLM-L6-H384-uncased",
        "architecture": "transformer",
        "experiment_phase": "controlled_backbone_benchmark",
        "learning_rate": 2e-5,
        "per_device_train_batch_size": 128,
        "gradient_accumulation_steps": 1,
        "effective_batch_size": 128,
        "weight_decay": 0.01,
        "dropout_prob": 0.25,
        "num_train_epochs": N_EPOCHS,
        "warmup_ratio": 0.03,
        "max_seq_len": 128,
        "head_hidden_dim": 256,
        "activation": "gelu",
        "focal_gamma": 2.0,
        "eval_batch_multiplier": 2,
    },
    "tinybert_bigru_attn": {
        "model_key": "tinybert_bigru_attn",
        "model_id": "huawei-noah/TinyBERT_General_6L_768D",
        "architecture": "tinybert_bigru_attention",
        "experiment_phase": "architecture_search",
        "learning_rate": 3e-5,
        "per_device_train_batch_size": 64,
        "gradient_accumulation_steps": 2,
        "effective_batch_size": 128,
        "weight_decay": 0.01,
        "dropout_prob": 0.25,
        "num_train_epochs": N_EPOCHS,
        "warmup_ratio": 0.04,
        "max_seq_len": 128,
        "head_hidden_dim": 256,
        "rnn_hidden_dim": 256,
        "rnn_layers": 1,
        "bidirectional": True,
        "attn_dim": 128,
        "activation": "gelu",
        "focal_gamma": 2.0,
        "eval_batch_multiplier": 2,
    },
    "albert_cnn": {
        "model_key": "albert_cnn",
        "model_id": "albert-base-v2",
        "architecture": "albert_cnn",
        "experiment_phase": "architecture_search",
        "learning_rate": 2.5e-5,
        "per_device_train_batch_size": 64,
        "gradient_accumulation_steps": 2,
        "effective_batch_size": 128,
        "weight_decay": 0.01,
        "dropout_prob": 0.25,
        "num_train_epochs": N_EPOCHS,
        "warmup_ratio": 0.04,
        "max_seq_len": 128,
        "head_hidden_dim": 256,
        "num_filters": 128,
        "kernel_sizes": [3, 5, 7],
        "activation": "gelu",
        "focal_gamma": 2.0,
        "eval_batch_multiplier": 2,
    },
}

RUN_MODEL_KEYS = ["distilbert", "minilm_l6", "tinybert_bigru_attn", "albert_cnn"]
BASELINE_LOSS_KEY = "weighted_focal"

if BASELINE_LOSS_KEY not in LOSS_ABLATION_GRID:
    raise ValueError(f"Baseline loss key {BASELINE_LOSS_KEY} must be one of {LOSS_ABLATION_GRID}")

LOSS_KEYS_BY_MODEL = {
    model_key: list(LOSS_ABLATION_GRID)
    if MODEL_REGISTRY[model_key]["experiment_phase"] == "controlled_backbone_benchmark"
    else [BASELINE_LOSS_KEY]
    for model_key in RUN_MODEL_KEYS
}

if SMOKE_MODE:
    BENCHMARK_SEEDS = list(SMOKE_SEEDS)
    RUN_MODEL_KEYS = [model_key for model_key in SMOKE_MODEL_KEYS if model_key in RUN_MODEL_KEYS]
    if not RUN_MODEL_KEYS:
        raise ValueError("Smoke mode is enabled but no model keys were selected.")

    active_smoke_losses = [loss_key for loss_key in SMOKE_LOSS_KEYS if loss_key in LOSS_ABLATION_GRID]
    if not active_smoke_losses:
        raise ValueError("Smoke mode is enabled but no valid loss keys were configured.")

    LOSS_KEYS_BY_MODEL = {model_key: list(active_smoke_losses) for model_key in RUN_MODEL_KEYS}

    EARLY_STOP_PATIENCE = 1
    LOG_EVERY_STEPS = 25

    for model_key in RUN_MODEL_KEYS:
        cfg = MODEL_REGISTRY[model_key]
        cfg["num_train_epochs"] = 1
        cfg["per_device_train_batch_size"] = int(min(cfg["per_device_train_batch_size"], 16))
        cfg["gradient_accumulation_steps"] = 1
        cfg["effective_batch_size"] = cfg["per_device_train_batch_size"]
        cfg["eval_batch_multiplier"] = 1

CONFIDENCE_THRESHOLDS = [0.5, 0.7, 0.8, 0.9]

DATA_DIR = resolve_data_dir(DATASET_VERSION)
RUN_OUTPUT_DIR = make_output_dir(DATASET_VERSION)
EVALUATION_OUTPUT_DIR = evaluation_dir(RUN_OUTPUT_DIR)

print(f"Data dir          : {DATA_DIR}")
print(f"Run output        : {RUN_OUTPUT_DIR}")
print(f"Evaluation dir    : {EVALUATION_OUTPUT_DIR}")
print(f"Benchmark seeds   : {BENCHMARK_SEEDS}")
print(f"Loss ablation grid: {list(LOSS_ABLATION_GRID)}")
if SMOKE_MODE:
    print("Smoke mode        : enabled")
    print(f"Smoke models      : {RUN_MODEL_KEYS}")
    print(f"Smoke losses      : {LOSS_KEYS_BY_MODEL}")
    print(f"Smoke sample cap  : {SMOKE_MAX_SAMPLES_PER_CLASS} per class")
else:
    print("Smoke mode        : disabled")

In [ ]:
def stratified_cap_by_label(df_split: pd.DataFrame, label_col: str, max_per_class: int, seed: int) -> pd.DataFrame:
    if max_per_class <= 0:
        return df_split.copy().reset_index(drop=True)

    sampled_parts = []
    for _, group in df_split.groupby(label_col):
        sampled_parts.append(group.sample(n=min(len(group), max_per_class), random_state=seed))

    sampled = pd.concat(sampled_parts, axis=0)
    sampled = sampled.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return sampled


df_train, df_val, df_test = load_data_splits(DATA_DIR, TEXT_COL, LABEL_COL)
if SMOKE_MODE:
    train_cap = int(max(8, SMOKE_MAX_SAMPLES_PER_CLASS))
    eval_cap = int(max(8, SMOKE_MAX_SAMPLES_PER_CLASS // 2))
    df_train = stratified_cap_by_label(df_train, LABEL_COL, max_per_class=train_cap, seed=BENCHMARK_SEEDS[0])
    df_val = stratified_cap_by_label(df_val, LABEL_COL, max_per_class=eval_cap, seed=BENCHMARK_SEEDS[0] + 1)
    df_test = stratified_cap_by_label(df_test, LABEL_COL, max_per_class=eval_cap, seed=BENCHMARK_SEEDS[0] + 2)

label_encoder, LABEL_NAMES = encode_labels(
    df_train=df_train,
    df_val=df_val,
    df_test=df_test,
    label_col=LABEL_COL,
    expected_classes=EXPECTED_CLASSES,
)
NUM_CLASSES = len(LABEL_NAMES)
SPLIT_SUMMARIES = build_split_summaries(df_train, df_val, df_test, LABEL_COL)
SPLIT_HYGIENE_EVIDENCE = build_split_hygiene_evidence(
    data_dir=DATA_DIR,
    split_summaries=SPLIT_SUMMARIES,
    df_train=df_train,
    df_val=df_val,
    df_test=df_test,
    text_col=TEXT_COL,
)

print(f"Dataset version : {DATASET_VERSION}")
if SMOKE_MODE:
    print("Dataset mode    : smoke subset")
print(f"Train           : {SPLIT_SUMMARIES['train']['size']:,}")
print(f"Validation      : {SPLIT_SUMMARIES['validation']['size']:,}")
print(f"Test            : {SPLIT_SUMMARIES['test']['size']:,}")
print(f"Classes ({NUM_CLASSES}): {LABEL_NAMES}")

for split_name in ["train", "validation", "test"]:
    print(f"\nLabel distribution ({split_name}):")
    distribution = SPLIT_SUMMARIES[split_name]["class_distribution"]
    for label_name, count in distribution.items():
        print(f"  {label_name:25s}: {count:,}")

print("\nSplit hygiene evidence summary:")
print(f"  zero_cross_split_overlap : {SPLIT_HYGIENE_EVIDENCE['zero_cross_split_overlap']}")
print(f"  overlap_counts           : {SPLIT_HYGIENE_EVIDENCE['cross_split_overlap_counts']}")
print(f"  metadata_sources_found   : {len(SPLIT_HYGIENE_EVIDENCE['metadata_sources'])}")
if SPLIT_HYGIENE_EVIDENCE["evidence_gaps"]:
    print("  evidence_gaps:")
    for gap in SPLIT_HYGIENE_EVIDENCE["evidence_gaps"]:
        print(f"    - {gap}")

In [ ]:
def seed_everything(seed: int = 42, deterministic: bool = True):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = bool(deterministic)
        torch.backends.cudnn.benchmark = not bool(deterministic)

    try:
        torch.use_deterministic_algorithms(bool(deterministic), warn_only=True)
    except Exception:
        pass


class WAFDataset(Dataset):
    def __init__(self, precomputed, labels):
        self.input_ids = precomputed["input_ids"]
        self.attention_mask = precomputed["attention_mask"]
        self.labels = labels.reset_index(drop=True)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.input_ids[idx], dtype=torch.long),
            "attention_mask": torch.tensor(self.attention_mask[idx], dtype=torch.long),
            "labels": torch.tensor(int(self.labels.iloc[idx]), dtype=torch.long),
        }


def compute_token_length_stats(texts, tokenizer, max_len: int, chunk_size: int = 4096) -> dict:
    lengths = []
    total = len(texts)
    for start in range(0, total, chunk_size):
        chunk = [str(text) for text in texts[start : start + chunk_size]]
        encoded = tokenizer(
            chunk,
            truncation=False,
            padding=False,
            add_special_tokens=True,
            return_attention_mask=False,
        )
        lengths.extend(len(ids) for ids in encoded["input_ids"])

    length_arr = np.asarray(lengths, dtype=np.int32)
    if length_arr.size == 0:
        return {
            "count": 0,
            "max_len": int(max_len),
            "mean": 0.0,
            "p50": 0.0,
            "p90": 0.0,
            "p95": 0.0,
            "p99": 0.0,
            "max": 0,
            "truncated_count": 0,
            "truncated_rate": 0.0,
        }

    truncated_count = int(np.sum(length_arr > max_len))
    return {
        "count": int(length_arr.size),
        "max_len": int(max_len),
        "mean": float(np.mean(length_arr)),
        "p50": float(np.percentile(length_arr, 50)),
        "p90": float(np.percentile(length_arr, 90)),
        "p95": float(np.percentile(length_arr, 95)),
        "p99": float(np.percentile(length_arr, 99)),
        "max": int(np.max(length_arr)),
        "truncated_count": truncated_count,
        "truncated_rate": float(truncated_count / max(length_arr.size, 1)),
    }


def build_truncation_evidence(tokenizer, max_len: int) -> dict:
    train_stats = compute_token_length_stats(df_train[TEXT_COL].tolist(), tokenizer, max_len=max_len)
    val_stats = compute_token_length_stats(df_val[TEXT_COL].tolist(), tokenizer, max_len=max_len)
    test_stats = compute_token_length_stats(df_test[TEXT_COL].tolist(), tokenizer, max_len=max_len)

    total_count = train_stats["count"] + val_stats["count"] + test_stats["count"]
    total_truncated = train_stats["truncated_count"] + val_stats["truncated_count"] + test_stats["truncated_count"]
    weighted_mean = (
        train_stats["mean"] * train_stats["count"]
        + val_stats["mean"] * val_stats["count"]
        + test_stats["mean"] * test_stats["count"]
    ) / max(total_count, 1)

    return {
        "max_len": int(max_len),
        "splits": {
            "train": train_stats,
            "validation": val_stats,
            "test": test_stats,
        },
        "overall": {
            "count": int(total_count),
            "truncated_count": int(total_truncated),
            "truncated_rate": float(total_truncated / max(total_count, 1)),
            "weighted_mean_tokens": float(weighted_mean),
        },
        "justification": (
            "MAX_LEN is kept only if truncation rate is operationally acceptable; "
            "this artifact makes truncation cost explicit for the current cleaned dataset."
        ),
    }


def preprocess_split(df_split: pd.DataFrame, tokenizer, max_len: int):
    encoded = tokenizer(
        list(df_split[TEXT_COL].astype(str).tolist()),
        truncation=True,
        max_length=max_len,
        padding=False,
        return_attention_mask=True,
    )
    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
    }


def tokenize_all_splits(df_train: pd.DataFrame, df_val: pd.DataFrame, df_test: pd.DataFrame, tokenizer, max_len: int):
    print("Pre-tokenizing train/validation/test splits ...")
    t0 = time.time()
    precomputed_train = preprocess_split(df_train, tokenizer, max_len)
    precomputed_val = preprocess_split(df_val, tokenizer, max_len)
    precomputed_test = preprocess_split(df_test, tokenizer, max_len)
    print(f"Tokenization done in {time.time() - t0:.1f}s")
    return precomputed_train, precomputed_val, precomputed_test


MODEL_RESOURCE_CACHE = {}


def prepare_model_resources(cfg: dict):
    model_key = cfg["model_key"]
    if model_key in MODEL_RESOURCE_CACHE:
        return MODEL_RESOURCE_CACHE[model_key]

    tokenizer = AutoTokenizer.from_pretrained(cfg["model_id"], use_fast=True)
    truncation_evidence = build_truncation_evidence(tokenizer=tokenizer, max_len=cfg["max_seq_len"])
    precomputed_train, precomputed_val, precomputed_test = tokenize_all_splits(
        df_train=df_train,
        df_val=df_val,
        df_test=df_test,
        tokenizer=tokenizer,
        max_len=cfg["max_seq_len"],
    )

    resources = {
        "tokenizer": tokenizer,
        "precomputed_train": precomputed_train,
        "precomputed_val": precomputed_val,
        "precomputed_test": precomputed_test,
        "truncation_evidence": truncation_evidence,
    }
    MODEL_RESOURCE_CACHE[model_key] = resources
    return resources


def build_dataloaders(resources: dict, cfg: dict, seed: int):
    tokenizer = resources["tokenizer"]
    collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True)
    generator = torch.Generator().manual_seed(int(seed))

    train_loader = DataLoader(
        WAFDataset(resources["precomputed_train"], df_train["label_id"]),
        batch_size=cfg["per_device_train_batch_size"],
        shuffle=True,
        generator=generator,
        collate_fn=collator,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )

    eval_bs = cfg["per_device_train_batch_size"] * cfg.get("eval_batch_multiplier", 2)

    val_loader = DataLoader(
        WAFDataset(resources["precomputed_val"], df_val["label_id"]),
        batch_size=eval_bs,
        shuffle=False,
        collate_fn=collator,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )

    test_loader = DataLoader(
        WAFDataset(resources["precomputed_test"], df_test["label_id"]),
        batch_size=eval_bs,
        shuffle=False,
        collate_fn=collator,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )

    return train_loader, val_loader, test_loader


def get_autocast_context():
    if DEVICE.type == "cuda" and CUDA_BF16:
        return autocast(device_type="cuda", dtype=torch.bfloat16)
    return nullcontext()


def build_optimizer(model, lr: float, weight_decay: float):
    no_decay = ["bias", "LayerNorm.weight", "layer_norm.weight"]
    grouped_parameters = [
        {
            "params": [
                p
                for n, p in model.named_parameters()
                if p.requires_grad and not any(nd in n for nd in no_decay)
            ],
            "weight_decay": weight_decay,
        },
        {
            "params": [
                p
                for n, p in model.named_parameters()
                if p.requires_grad and any(nd in n for nd in no_decay)
            ],
            "weight_decay": 0.0,
        },
    ]
    return torch.optim.AdamW(grouped_parameters, lr=lr)


def train_one_epoch(model, dataloader, optimizer, scheduler, criterion, cfg):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    n_steps = len(dataloader)
    accum_steps = cfg["gradient_accumulation_steps"]

    for step, batch in enumerate(dataloader, start=1):
        ids = batch["input_ids"].to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
        mask = batch["attention_mask"].to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
        labels = batch["labels"].to(DEVICE, non_blocking=(DEVICE.type == "cuda"))

        with get_autocast_context():
            out = model(input_ids=ids, attention_mask=mask)
            loss = criterion(out["logits"], labels) / accum_steps

        if not torch.isfinite(loss):
            raise RuntimeError(f"Diverged at batch {step}: loss={loss.item():.6f}")

        loss.backward()
        total_loss += loss.item() * accum_steps

        if (step % accum_steps == 0) or (step == n_steps):
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        if (step % LOG_EVERY_STEPS == 0) or (step == n_steps):
            print(f"  step {step:>5,}/{n_steps:,} | loss {total_loss / step:.4f}")

    epoch_loss = total_loss / max(n_steps, 1)
    epoch_lr = float(scheduler.get_last_lr()[0])
    return epoch_loss, epoch_lr


def run_single_seed_experiment(cfg: dict, loss_key: str, seed: int, variant_dir: Path, resources: dict):
    seed_everything(seed, deterministic=True)

    model_key = cfg["model_key"]
    seed_dir = seed_run_dir(variant_dir, seed)
    ckpt_dir = checkpoint_dir(seed_dir)
    best_ckpt_path = ckpt_dir / f"best_{model_key}_{loss_key}_seed{int(seed):04d}.pt"

    print("\n" + "=" * 108)
    print(f"Model={model_key} | loss={loss_key} | seed={seed} | phase={cfg['experiment_phase']}")
    print("=" * 108)

    train_loader, val_loader, test_loader = build_dataloaders(resources=resources, cfg=cfg, seed=seed)
    print(f"Train batches: {len(train_loader):,} | Val batches: {len(val_loader):,} | Test batches: {len(test_loader):,}")

    class_weights_np = compute_class_weights(df_train["label_id"].to_numpy(dtype=np.int64, copy=False), LABEL_NAMES)
    class_weights = torch.tensor(class_weights_np, dtype=torch.float32, device=DEVICE)
    criterion, loss_metadata = build_loss(
        loss_key=loss_key,
        class_weights=class_weights,
        gamma=cfg.get("focal_gamma", 2.0),
    )

    effective_class_weights = {
        label_name: float(class_weights_np[idx]) for idx, label_name in enumerate(LABEL_NAMES)
    }

    config_metadata = {
        "model_key": model_key,
        "model_id": cfg["model_id"],
        "architecture": cfg["architecture"],
        "architecture_family": infer_architecture_family(cfg["architecture"]),
        "head_type": infer_head_type(cfg["architecture"]),
        "experiment_phase": cfg["experiment_phase"],
        "seed": int(seed),
        "dataset_version": DATASET_VERSION,
        "max_seq_len": cfg["max_seq_len"],
        "max_grad_norm": MAX_GRAD_NORM,
        "checkpoint_selection_rule": CHECKPOINT_SELECTION_RULE,
        "model_selection_rule": MODEL_SELECTION_RULE,
        "loss": loss_metadata,
        "effective_class_weights": effective_class_weights,
        "training_hyperparameters": {
            "learning_rate": cfg["learning_rate"],
            "weight_decay": cfg["weight_decay"],
            "num_train_epochs": cfg["num_train_epochs"],
            "gradient_accumulation_steps": cfg["gradient_accumulation_steps"],
            "per_device_train_batch_size": cfg["per_device_train_batch_size"],
            "effective_batch_size": cfg["effective_batch_size"],
            "warmup_ratio": cfg["warmup_ratio"],
        },
    }
    save_json(seed_dir / "config_metadata.json", config_metadata)
    save_json(seed_dir / "truncation_evidence.json", resources["truncation_evidence"])

    model = build_model(cfg, NUM_CLASSES, DEVICE)
    optimizer = build_optimizer(model, cfg["learning_rate"], cfg["weight_decay"])

    steps_per_epoch = max(math.ceil(len(train_loader) / cfg["gradient_accumulation_steps"]), 1)
    total_steps = steps_per_epoch * cfg["num_train_epochs"]
    warmup_steps = int(cfg["warmup_ratio"] * total_steps)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    best_val_macro_f1 = -1.0
    best_val_loss = float("inf")
    best_epoch = 0
    best_state = None
    best_val_logits = None
    best_val_labels = None
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, cfg["num_train_epochs"] + 1):
        print(f"\nEpoch {epoch}/{cfg['num_train_epochs']}")
        train_loss, learning_rate = train_one_epoch(
            model=model,
            dataloader=train_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            criterion=criterion,
            cfg=cfg,
        )

        val_loss, val_logits, val_labels = collect_logits_labels_loss(
            model=model,
            dataloader=val_loader,
            criterion=criterion,
            device=DEVICE,
            autocast_context_fn=get_autocast_context,
        )
        val_metrics = evaluate_from_logits(val_logits, val_labels, n_bins=ECE_N_BINS)

        print(
            f"  train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
            f"val_macro_f1={val_metrics['macro_f1']:.4f} | val_acc={val_metrics['accuracy']:.4f} | lr={learning_rate:.2e}"
        )

        history.append(
            {
                "epoch": int(epoch),
                "learning_rate": float(learning_rate),
                "train_loss": float(train_loss),
                "val_loss": float(val_loss),
                "val_accuracy": float(val_metrics["accuracy"]),
                "val_macro_f1": float(val_metrics["macro_f1"]),
                "val_weighted_f1": float(val_metrics["weighted_f1"]),
                "val_ece": float(val_metrics["ece"]),
                "val_nll": float(val_metrics["nll"]),
            }
        )

        improved = (val_metrics["macro_f1"] > best_val_macro_f1 + 1e-12) or (
            abs(val_metrics["macro_f1"] - best_val_macro_f1) <= 1e-12 and val_loss < best_val_loss
        )

        if improved:
            best_val_macro_f1 = float(val_metrics["macro_f1"])
            best_val_loss = float(val_loss)
            best_epoch = int(epoch)
            best_state = deepcopy(model.state_dict())
            best_val_logits = val_logits
            best_val_labels = val_labels
            torch.save(
                {
                    "epoch": int(epoch),
                    "model_state_dict": best_state,
                    "selection_metric": "val_macro_f1",
                    "val_macro_f1": float(best_val_macro_f1),
                    "val_loss": float(best_val_loss),
                    "cfg": cfg,
                    "loss_key": loss_key,
                    "seed": int(seed),
                },
                best_ckpt_path,
            )
            epochs_without_improvement = 0
            print(f"  new best checkpoint saved: {best_ckpt_path.name}")
        else:
            epochs_without_improvement += 1
            print(f"  no improvement ({epochs_without_improvement}/{EARLY_STOP_PATIENCE})")
            if epochs_without_improvement >= EARLY_STOP_PATIENCE:
                print("  early stopping triggered")
                break

    if best_state is None or best_val_logits is None or best_val_labels is None:
        raise RuntimeError("Training did not produce a best checkpoint with validation logits.")

    model.load_state_dict(best_state)
    temperature = fit_temperature_scaling(best_val_logits, best_val_labels, device=DEVICE)

    _, test_uncal_logits, test_uncal_labels = collect_logits_labels_loss(
        model=model,
        dataloader=test_loader,
        criterion=criterion,
        device=DEVICE,
        autocast_context_fn=get_autocast_context,
    )

    val_uncal_metrics = evaluate_from_logits(best_val_logits, best_val_labels, n_bins=ECE_N_BINS)
    test_uncal_metrics = evaluate_from_logits(test_uncal_logits, test_uncal_labels, n_bins=ECE_N_BINS)
    val_cal_metrics = evaluate_from_logits(best_val_logits / temperature, best_val_labels, n_bins=ECE_N_BINS)
    test_cal_metrics = evaluate_from_logits(test_uncal_logits / temperature, test_uncal_labels, n_bins=ECE_N_BINS)

    save_confusion_matrix_artifacts(
        labels=test_uncal_labels,
        preds=test_uncal_metrics["preds"],
        label_names=LABEL_NAMES,
        csv_path=seed_dir / "confusion_matrix.csv",
        png_path=seed_dir / "confusion_matrix.png",
        title=f"{model_key} test confusion matrix | {loss_key} | seed {seed}",
    )

    save_reliability_diagram_artifacts(
        probs=test_uncal_metrics["probs"],
        labels=test_uncal_labels,
        csv_path=seed_dir / "reliability_uncalibrated.csv",
        png_path=seed_dir / "reliability_uncalibrated.png",
        n_bins=ECE_N_BINS,
        title=f"{model_key} uncalibrated reliability",
    )
    save_reliability_diagram_artifacts(
        probs=test_cal_metrics["probs"],
        labels=test_uncal_labels,
        csv_path=seed_dir / "reliability_calibrated.csv",
        png_path=seed_dir / "reliability_calibrated.png",
        n_bins=ECE_N_BINS,
        title=f"{model_key} calibrated reliability",
    )

    top_label_uncal_df = top_label_calibration_frame(test_uncal_metrics["probs"], test_uncal_labels, LABEL_NAMES)
    top_label_cal_df = top_label_calibration_frame(test_cal_metrics["probs"], test_uncal_labels, LABEL_NAMES)
    save_csv(top_label_uncal_df, seed_dir / "top_label_calibration_uncalibrated.csv", index=False)
    save_csv(top_label_cal_df, seed_dir / "top_label_calibration_calibrated.csv", index=False)

    security_views = threshold_security_summary(
        labels=test_uncal_labels,
        preds=test_uncal_metrics["preds"],
        probs=test_uncal_metrics["probs"],
        label_names=LABEL_NAMES,
        normal_label="Normal",
    )
    attack_to_normal_df = security_views["attack_to_normal_by_class"]
    if attack_to_normal_df.empty:
        attack_to_normal_df = pd.DataFrame(
            columns=[
                "attack_label",
                "attack_label_id",
                "total_true_samples",
                "predicted_as_normal",
                "false_negative_rate_to_normal",
            ]
        )

    confidence_band_df = confidence_band_summary_frame(
        labels=test_uncal_labels,
        preds=test_uncal_metrics["preds"],
        probs=test_uncal_metrics["probs"],
    )
    threshold_recall_df = per_class_recall_at_threshold_frame(
        labels=test_uncal_labels,
        preds=test_uncal_metrics["preds"],
        probs=test_uncal_metrics["probs"],
        label_names=LABEL_NAMES,
        thresholds=CONFIDENCE_THRESHOLDS,
    )

    save_csv(attack_to_normal_df, seed_dir / "attack_to_normal_fn.csv", index=False)
    save_csv(confidence_band_df, seed_dir / "confidence_band_summary.csv", index=False)
    save_csv(threshold_recall_df, seed_dir / "per_class_recall_at_threshold.csv", index=False)
    per_class_metrics = compute_per_class_metrics(
        labels=test_uncal_labels,
        preds=test_uncal_metrics["preds"],
        label_names=LABEL_NAMES,
    )
    save_json(seed_dir / "per_class_metrics.json", per_class_metrics)
    save_json(
        seed_dir / "security_summary.json",
        {
            "normal_false_positive": security_views["normal_false_positive"],
            "attack_escape_total": security_views["attack_escape_total"],
            "confidence_thresholds": CONFIDENCE_THRESHOLDS,
            "confidence_bands": [
                {"name": "LOW", "range": "[0.0, 0.5)"},
                {"name": "MEDIUM", "range": "[0.5, 0.8)"},
                {"name": "HIGH", "range": "[0.8, 1.0]"},
            ],
        },
    )

    save_numpy_artifacts(
        seed_dir / "validation_outputs.npz",
        logits=best_val_logits,
        labels=best_val_labels,
        preds=val_uncal_metrics["preds"],
        probs=val_uncal_metrics["probs"],
        calibrated_probs=val_cal_metrics["probs"],
    )
    save_numpy_artifacts(
        seed_dir / "test_outputs.npz",
        logits=test_uncal_logits,
        labels=test_uncal_labels,
        preds=test_uncal_metrics["preds"],
        probs=test_uncal_metrics["probs"],
        calibrated_probs=test_cal_metrics["probs"],
    )

    sample_batch = next(iter(test_loader))
    latency_ms = estimate_inference_latency_ms(
        model=model,
        sample_batch=sample_batch,
        device=DEVICE,
        warmup_steps=3,
        measure_steps=20,
        autocast_context_fn=get_autocast_context,
    )
    model_size_mb = model_size_megabytes(model)

    deployment_payload = {
        "model_size_mb": float(model_size_mb),
        "inference_latency_ms": float(latency_ms),
        "latency_protocol": {
            "batch_size": int(sample_batch["input_ids"].shape[0]),
            "sequence_length": int(sample_batch["input_ids"].shape[1]),
            "warmup_steps": 3,
            "measure_steps": 20,
            "device": str(DEVICE),
            "autocast_bf16": bool(CUDA_BF16 and DEVICE.type == "cuda"),
        },
    }
    save_json(seed_dir / "deployment_metrics.json", deployment_payload)

    calibration_payload = {
        "method": "temperature_scaling",
        "fit_split": "validation",
        "temperature": float(temperature),
        "val_ece_uncalibrated": float(val_uncal_metrics["ece"]),
        "val_ece_calibrated": float(val_cal_metrics["ece"]),
        "test_ece_uncalibrated": float(test_uncal_metrics["ece"]),
        "test_ece_calibrated": float(test_cal_metrics["ece"]),
        "val_nll_uncalibrated": float(val_uncal_metrics["nll"]),
        "val_nll_calibrated": float(val_cal_metrics["nll"]),
        "test_nll_uncalibrated": float(test_uncal_metrics["nll"]),
        "test_nll_calibrated": float(test_cal_metrics["nll"]),
    }
    save_json(seed_dir / "calibration.json", calibration_payload)
    save_json(seed_dir / "train_history.json", history)

    summary_metrics = {
        "model_key": model_key,
        "model_id": cfg["model_id"],
        "architecture": cfg["architecture"],
        "architecture_family": infer_architecture_family(cfg["architecture"]),
        "head_type": infer_head_type(cfg["architecture"]),
        "experiment_phase": cfg["experiment_phase"],
        "loss_key": loss_key,
        "seed": int(seed),
        "best_epoch": int(best_epoch),
        "checkpoint_selection_rule": CHECKPOINT_SELECTION_RULE,
        "best_val_macro_f1": float(best_val_macro_f1),
        "best_val_loss": float(best_val_loss),
        "temperature": float(temperature),
        "trainable_params": int(sum(p.numel() for p in model.parameters() if p.requires_grad)),
        "val_accuracy": float(val_uncal_metrics["accuracy"]),
        "val_macro_f1": float(val_uncal_metrics["macro_f1"]),
        "val_weighted_f1": float(val_uncal_metrics["weighted_f1"]),
        "val_ece_uncalibrated": float(val_uncal_metrics["ece"]),
        "val_ece_calibrated": float(val_cal_metrics["ece"]),
        "val_nll_uncalibrated": float(val_uncal_metrics["nll"]),
        "val_nll_calibrated": float(val_cal_metrics["nll"]),
        "test_accuracy": float(test_uncal_metrics["accuracy"]),
        "test_macro_f1": float(test_uncal_metrics["macro_f1"]),
        "test_weighted_f1": float(test_uncal_metrics["weighted_f1"]),
        "test_ece_uncalibrated": float(test_uncal_metrics["ece"]),
        "test_ece_calibrated": float(test_cal_metrics["ece"]),
        "test_nll_uncalibrated": float(test_uncal_metrics["nll"]),
        "test_nll_calibrated": float(test_cal_metrics["nll"]),
        "model_size_mb": float(model_size_mb),
        "inference_latency_ms": float(latency_ms),
        "normal_false_positive_rate": float(security_views["normal_false_positive"]["normal_false_positive_rate"]),
        "attack_escape_rate": float(security_views["attack_escape_total"]["attack_escape_rate"]),
    }
    save_json(seed_dir / "summary_metrics.json", summary_metrics)

    return summary_metrics


def aggregate_seed_summaries(seed_summaries: list[dict]):
    seed_df = pd.DataFrame(seed_summaries).sort_values(by="seed").reset_index(drop=True)
    numeric_columns = [
        col
        for col in seed_df.columns
        if pd.api.types.is_numeric_dtype(seed_df[col]) and col not in {"seed"}
    ]

    aggregate = {}
    for col in numeric_columns:
        values = np.asarray(seed_df[col].to_numpy(dtype=np.float64), dtype=np.float64)
        aggregate[f"{col}_mean"] = float(np.mean(values))
        aggregate[f"{col}_std"] = float(np.std(values, ddof=0))

    return seed_df, aggregate


print("Training utilities are ready.")

In [ ]:
seed_everything(BENCHMARK_SEEDS[0], deterministic=True)

all_variant_rows = []
selected_variant_rows = []
model_selection_manifest = {}
model_truncation_overview = {}

for model_key in RUN_MODEL_KEYS:
    cfg = MODEL_REGISTRY[model_key]
    model_dir = model_run_dir(RUN_OUTPUT_DIR, model_key)
    resources = prepare_model_resources(cfg)
    save_json(model_dir / "truncation_evidence.json", resources["truncation_evidence"])
    model_truncation_overview[model_key] = resources["truncation_evidence"]["overall"]

    variant_rows = []
    for loss_key in LOSS_KEYS_BY_MODEL[model_key]:
        variant_dir = loss_variant_dir(model_dir, loss_key)
        seed_summaries = []

        for seed in BENCHMARK_SEEDS:
            result = run_single_seed_experiment(
                cfg=cfg,
                loss_key=loss_key,
                seed=seed,
                variant_dir=variant_dir,
                resources=resources,
            )
            seed_summaries.append(result)

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        seed_df, aggregate = aggregate_seed_summaries(seed_summaries)
        save_csv(seed_df, variant_dir / "seed_summaries.csv", index=False)

        aggregate_payload = {
            "model_key": model_key,
            "loss_key": loss_key,
            "n_seeds": len(BENCHMARK_SEEDS),
            "seed_list": BENCHMARK_SEEDS,
            "selection_metric": "validation_macro_f1_mean",
            "selection_sort_keys": ["val_macro_f1_mean"],
            "selection_note": "Selection is validation-driven. Test metrics are final evidence only.",
            **aggregate,
        }
        save_json(variant_dir / "aggregate_summary.json", aggregate_payload)

        variant_row = {
            "model_key": model_key,
            "loss_key": loss_key,
            "architecture": cfg["architecture"],
            "architecture_family": infer_architecture_family(cfg["architecture"]),
            "head_type": infer_head_type(cfg["architecture"]),
            "experiment_phase": cfg["experiment_phase"],
            "n_seeds": len(BENCHMARK_SEEDS),
            **aggregate,
        }
        variant_rows.append(variant_row)
        all_variant_rows.append(variant_row)

    variant_df = pd.DataFrame(variant_rows).sort_values(
        by=["val_macro_f1_mean"],
        ascending=[False],
    ).reset_index(drop=True)
    save_csv(variant_df, model_dir / "loss_variant_aggregates.csv", index=False)

    selected_row = variant_df.iloc[0].to_dict()
    selected_row["model_selection_rule"] = MODEL_SELECTION_RULE
    selected_variant_rows.append(selected_row)

    model_selection_manifest[model_key] = {
        "selected_loss_key": str(selected_row["loss_key"]),
        "available_loss_keys": list(LOSS_KEYS_BY_MODEL[model_key]),
        "selection_rule": MODEL_SELECTION_RULE,
        "selection_metric": "validation_macro_f1_mean",
        "selection_sort_keys": ["val_macro_f1_mean"],
        "experiment_phase": cfg["experiment_phase"],
    }

all_variants_df = pd.DataFrame(all_variant_rows).sort_values(
    by=["experiment_phase", "model_key", "val_macro_f1_mean"],
    ascending=[True, True, False],
).reset_index(drop=True)

comparison_df = pd.DataFrame(selected_variant_rows).sort_values(
    by=["experiment_phase", "model_key"],
    ascending=[True, True],
).reset_index(drop=True)

summary_cols = [
    "model_key",
    "experiment_phase",
    "loss_key",
    "architecture",
    "head_type",
    "n_seeds",
    "val_macro_f1_mean",
    "val_macro_f1_std",
    "test_macro_f1_mean",
    "test_macro_f1_std",
    "test_ece_uncalibrated_mean",
    "test_ece_calibrated_mean",
    "test_nll_uncalibrated_mean",
    "test_nll_calibrated_mean",
    "inference_latency_ms_mean",
    "model_size_mb_mean",
    "normal_false_positive_rate_mean",
    "attack_escape_rate_mean",
]

display(
    comparison_df[summary_cols].style.format(
        {
            "val_macro_f1_mean": "{:.4f}",
            "val_macro_f1_std": "{:.4f}",
            "test_macro_f1_mean": "{:.4f}",
            "test_macro_f1_std": "{:.4f}",
            "test_ece_uncalibrated_mean": "{:.4f}",
            "test_ece_calibrated_mean": "{:.4f}",
            "test_nll_uncalibrated_mean": "{:.4f}",
            "test_nll_calibrated_mean": "{:.4f}",
            "inference_latency_ms_mean": "{:.3f}",
            "model_size_mb_mean": "{:.2f}",
            "normal_false_positive_rate_mean": "{:.4f}",
            "attack_escape_rate_mean": "{:.4f}",
        }
    )
)

save_csv(all_variants_df, RUN_OUTPUT_DIR / "all_loss_variant_aggregates.csv", index=False)
save_csv(comparison_df, RUN_OUTPUT_DIR / "comparison_results.csv", index=False)

run_manifest = {
    "dataset_version": DATASET_VERSION,
    "text_col": TEXT_COL,
    "label_col": LABEL_COL,
    "label_names": LABEL_NAMES,
    "seed_list": BENCHMARK_SEEDS,
    "ece_n_bins": ECE_N_BINS,
    "max_grad_norm": MAX_GRAD_NORM,
    "run_model_keys": RUN_MODEL_KEYS,
    "loss_keys_by_model": LOSS_KEYS_BY_MODEL,
    "checkpoint_selection_rule": CHECKPOINT_SELECTION_RULE,
    "model_selection_rule": MODEL_SELECTION_RULE,
    "variant_selection_sort_keys": ["val_macro_f1_mean"],
    "selection_guardrail": "No model winner is selected from test metrics.",
    "split_summaries": SPLIT_SUMMARIES,
    "split_hygiene_evidence": SPLIT_HYGIENE_EVIDENCE,
    "model_truncation_overview": model_truncation_overview,
    "model_selection_manifest": model_selection_manifest,
    "models": {
        model_key: {
            "model_id": MODEL_REGISTRY[model_key]["model_id"],
            "architecture": MODEL_REGISTRY[model_key]["architecture"],
            "architecture_family": infer_architecture_family(MODEL_REGISTRY[model_key]["architecture"]),
            "head_type": infer_head_type(MODEL_REGISTRY[model_key]["architecture"]),
            "experiment_phase": MODEL_REGISTRY[model_key]["experiment_phase"],
            "max_seq_len": MODEL_REGISTRY[model_key]["max_seq_len"],
            "loss_keys": LOSS_KEYS_BY_MODEL[model_key],
        }
        for model_key in RUN_MODEL_KEYS
    },
    "created_at": datetime.now(timezone.utc).isoformat(),
    "run_output_dir": str(RUN_OUTPUT_DIR),
    "evaluation_output_dir": str(EVALUATION_OUTPUT_DIR),
}
save_json(RUN_OUTPUT_DIR / "run_manifest.json", run_manifest)

print("Saved:")
print(f"- {RUN_OUTPUT_DIR / 'all_loss_variant_aggregates.csv'}")
print(f"- {RUN_OUTPUT_DIR / 'comparison_results.csv'}")
print(f"- {RUN_OUTPUT_DIR / 'run_manifest.json'}")
print(f"- {EVALUATION_OUTPUT_DIR}")
print("Per-model artifacts are saved under: model/loss_<loss_key>/seed_<seed>/")
print("Model selection is validation-driven; test metrics remain report-only evidence.")